# 03b – Preprocessing v2 (erweiterte Features)

**Vergleich zu v1:** [docs/10_PREPROCESSING_V2.md](../docs/10_PREPROCESSING_V2.md) · Original: `03_preprocessing.ipynb`

| Output v2 | v1 (alt) |
|-----------|----------|
| `train_labeled_v2.parquet` | `train_labeled.parquet` |
| `test_features_v2.parquet` | `test_features.parquet` |

Neu: Score-Lags, `score_persist7` als Feature, Regional-Score-Stats, 91-Tage-Test-Aggregate.

→ danach **`04b_modeling_v2.ipynb`**

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_repo = Path("/content/DataMining_Final-Project")
if not (_repo / "scripts" / "notebook_init.py").exists():
    try:
        import google.colab  # noqa: F401
        import subprocess
        subprocess.run(
            ["git", "clone", "--branch", "main",
             "https://github.com/jspldrch/DataMining_Final-Project.git", str(_repo)],
            check=True,
        )
    except ImportError:
        pass
for _p in (Path.cwd(), Path.cwd().parent, _repo):
    if (_p / "scripts" / "notebook_init.py").exists() and str(_p.resolve()) not in sys.path:
        sys.path.insert(0, str(_p.resolve()))

from scripts.notebook_init import setup
from scripts.project_env import load_script

env = setup()
PROJECT_ROOT = env["project_root"]
TRAIN_PATH = env["train_path"]
TEST_PATH = env["test_path"]
PROCESSED_DIR = env["processed_dir"]
MODE = env["mode"]

_fv2 = load_script("features_v2", PROJECT_ROOT / "scripts" / "features_v2.py")
feature_columns_v2 = _fv2.feature_columns_v2
build_features_v2 = _fv2.build_features_v2

print("Setup OK | MODE:", MODE, "| Features v2:", len(feature_columns_v2()))

In [ ]:
path_train_v2 = PROCESSED_DIR / "train_labeled_v2.parquet"
path_test_v2 = PROCESSED_DIR / "test_features_v2.parquet"

if MODE == "full":
    _ps2 = load_script("preprocess_v2", PROJECT_ROOT / "scripts" / "preprocess_streaming_v2.py")
    print("Streaming v2 (pro Region) — ~20–40 Min. …")
    stats = _ps2.preprocess_by_region_v2(
        TRAIN_PATH, TEST_PATH, path_train_v2, path_test_v2, chunk_size=500_000
    )
    print("Fertig:", stats)
else:
    _ps2 = load_script("preprocess_v2", PROJECT_ROOT / "scripts" / "preprocess_streaming_v2.py")
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    train_labeled, test_feat = _ps2.preprocess_panel_v2(train, test)
    train_labeled.to_parquet(path_train_v2, index=False)
    test_feat.to_parquet(path_test_v2, index=False)
    print(f"Gespeichert: {len(train_labeled):,} train | {len(test_feat):,} test")

In [ ]:
import pyarrow.parquet as pq

n_train = pq.read_metadata(path_train_v2).num_rows
n_test = pq.read_metadata(path_test_v2).num_rows
print(f"Parquet v2: train {n_train:,} | test {n_test:,}")

head = pd.read_parquet(path_train_v2)
v2_only = [c for c in head.columns if c.startswith(("score_lag", "region_score", "test91_"))]
print(f"Neue Spalten (Beispiel): {v2_only[:8]} … ({len(v2_only)} v2-spezifisch sichtbar)")
head.head(3)